# syft-bg Integration Tests

1. **Config & Status basics** — init, inspect config, check status
2. **Misconfigured `email_approve`** — missing `gcp_project_id` → graceful error
3. **Start / Stop / Restart** — verify PID lifecycle
4. **Auto-approve job via API** — `auto_approve_job()`, follow-up auto-approval, list/remove rules
5. **Job repr for auto-approved jobs (DS side)** — verify approval_method shows `auto`
6. **Invalid DO token at startup** — service reports ERROR status

### Prerequisites
- `.env` with `DO_EMAIL`, `DS_EMAIL`, `GCP_PROJECT_ID`, `APP_CREDENTIALS`, `TOKEN_DS`, `DO_CREDENTIALS`, `DO_TOKEN`

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import time
import json
import shutil
import tempfile
import uuid
from pathlib import Path

# Load .env file
for line in Path(".env").read_text().splitlines():
    if line.strip() and not line.startswith("#"):
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())

DO_EMAIL = os.environ["DO_EMAIL"]
DS_EMAIL = os.environ["DS_EMAIL"]
GCP_PROJECT_ID = os.environ["GCP_PROJECT_ID"]

APP_CREDENTIALS = Path(os.environ["APP_CREDENTIALS"])
TOKEN_DS = Path(os.environ["TOKEN_DS"])
DO_CREDENTIALS = Path(os.environ["DO_CREDENTIALS"])
DO_TOKEN = Path(os.environ["DO_TOKEN"])

assert APP_CREDENTIALS.exists(), "App credentials missing"
assert TOKEN_DS.exists(), "DS credentials missing"
assert DO_CREDENTIALS.exists(), "DO credentials missing"
assert DO_TOKEN.exists(), "DO token missing"

In [3]:
import syft_client as sc

do_client = sc.login_do(email=DO_EMAIL, token_path=str(DO_TOKEN))
ds_client = sc.login_ds(email=DS_EMAIL, token_path=str(TOKEN_DS))

Ensured directories exist: /Users/bitsofsteve/SyftBox_stephen@openmined.org/stephen@openmined.org/app_data/job
🔑 Logging in as stephen@openmined.org...
   Environment: Python
Found full checkpoint with 93 files, restoring...
Found 1 incremental checkpoints, applying...
Found rolling state with 18 events, applying...
Uploading rolling state with 18 events...

✅ Logged in successfully!
   SyftBox folder : /Users/bitsofsteve/SyftBox_stephen@openmined.org
   Version        : 0.1.112

👥 1 peer(s) restored from previous session.
   Run client.sync() to pull the latest updates.
🔑 Logging in as cyberlexix@gmail.com...
   Environment: Python

✅ Logged in successfully!
   SyftBox folder : /Users/bitsofsteve/SyftBox_cyberlexix@gmail.com
   Version        : 0.1.112

👥 1 peer(s) restored from previous session.
   Run client.sync() to pull the latest updates.


In [4]:
ds_client.add_peer(DO_EMAIL)
do_client.load_peers()
do_client.approve_peer_request(DS_EMAIL, peer_must_exist=False)

try:
    do_client.create_dataset(name="testdata", mock_path="data/mock.txt",
                             private_path="data/private.txt", users=[DS_EMAIL])
except FileExistsError:
    print("Dataset 'testdata' already exists")

ℹ️  Already have a connection with stephen@openmined.org (state: accepted).
   Use force=True to resend the request.
Peer cyberlexix@gmail.com is already approved, skipping


Failed to create dataset, cleaning up


Dataset 'testdata' already exists


---
## Test 1: Config & Status Basics

In [5]:
import syft_bg
from syft_bg.services.base import ServiceStatus

syft_bg.reset()
syft_bg.init(do_email=DO_EMAIL, token_path=str(DO_TOKEN),
             settings={"email_approve": {"gcp_project_id": GCP_PROJECT_ID}})

Error stopping service notify: notify is not running
Error stopping service approve: approve is not running
Error stopping service email_approve: email_approve is not running
Error stopping service sync: sync is not running
Reset complete: uninstalled systemd units, stopped services, cleared state, config, logs, and PID files.
Stored token at /Users/bitsofsteve/.syft-bg/token.json
Config saved to /Users/bitsofsteve/.syft-bg/config.yaml


In [6]:
syft_bg.config

SyftBgConfig(do_email='stephen@openmined.org', syftbox_root='/Users/bitsofsteve/SyftBox_stephen@openmined.org', credentials_path=PosixPath('/Users/bitsofsteve/.syft-bg/credentials.json'), gmail_token_path=PosixPath('/Users/bitsofsteve/.syft-bg/token.json'), drive_token_path=PosixPath('/Users/bitsofsteve/code/syft-client/notebooks/beach/internal/token_do.json'), notify=NotifyConfig(do_email='stephen@openmined.org', syftbox_root=PosixPath('/Users/bitsofsteve/SyftBox_stephen@openmined.org'), drive_token_path=PosixPath('/Users/bitsofsteve/code/syft-client/notebooks/beach/internal/token_do.json'), gmail_token_path=PosixPath('/Users/bitsofsteve/.syft-bg/token.json'), credentials_path=PosixPath('/Users/bitsofsteve/.syft-bg/credentials.json'), notify_state_path=PosixPath('/Users/bitsofsteve/.syft-bg/notify/state.json'), sync_state_path=PosixPath('/Users/bitsofsteve/.syft-bg/sync/state.json'), interval=30, monitor_jobs=True, monitor_peers=True), approve=AutoApproveConfig(do_email='stephen@openmined.org', syftbox_root=PosixPath('/Users/bitsofsteve/SyftBox_stephen@openmined.org'), drive_token_path=PosixPath('/Users/bitsofsteve/code/syft-client/notebooks/beach/internal/token_do.json'), gmail_token_path=PosixPath('/Users/bitsofsteve/.syft-bg/token.json'), approve_state_path=PosixPath('/Users/bitsofsteve/.syft-bg/approve/state.json'), notify_state_path=PosixPath('/Users/bitsofsteve/.syft-bg/notify/state.json'), interval=5, auto_approvals=AutoApprovalsConfig(enabled=True, objects={}), peers=PeerApprovalConfig(enabled=False, approved_domains=[], auto_share_datasets=[], auto_approve_emails=[])), email_approve=EmailApproveConfig(do_email='stephen@openmined.org', syftbox_root=PosixPath('/Users/bitsofsteve/SyftBox_stephen@openmined.org'), gmail_token_path=PosixPath('/Users/bitsofsteve/.syft-bg/token.json'), gcp_project_id='cyberlexix', pubsub_topic=None, pubsub_subscription=None, email_approve_state_path=PosixPath('/Users/bitsofsteve/.syft-bg/email_approve/state.json'), notify_state_path=PosixPath('/Users/bitsofsteve/.syft-bg/notify/state.json')), sync=SyncConfig(interval=10, max_retries=3, retry_backoff=2.0, do_email='stephen@openmined.org', syftbox_root=PosixPath('/Users/bitsofsteve/SyftBox_stephen@openmined.org'), drive_token_path=PosixPath('/Users/bitsofsteve/code/syft-client/notebooks/beach/internal/token_do.json'), sync_state_path=PosixPath('/Users/bitsofsteve/.syft-bg/sync/state.json')))

In [7]:
status = syft_bg.status
for name, info in status.service_infos.items():
    assert info.status.value in ("stopped", "unknown"), f"{name} should be stopped, got {info.status}"
print("All services stopped as expected")

All services stopped as expected


---
## Test 2: Misconfigured `email_approve` (no `gcp_project_id`)

In [8]:
syft_bg.reset()
syft_bg.init(do_email=DO_EMAIL, token_path=str(DO_TOKEN))

syft_bg.ensure_running(services=["email_approve"])
time.sleep(3)  # wait for subprocess to fail

ea_info = syft_bg.status.service_infos["email_approve"]
assert ea_info.status == ServiceStatus.ERROR, f"Expected ERROR, got {ea_info.status}"
assert ea_info.error and "gcp_project_id" in ea_info.error
print(f"email_approve: {ea_info.status.value}")
print(f"Error confirms missing gcp_project_id")

Error stopping service notify: notify is not running
Error stopping service approve: approve is not running
Error stopping service email_approve: email_approve is not running
Error stopping service sync: sync is not running
Reset complete: uninstalled systemd units, stopped services, cleared state, config, logs, and PID files.
Stored token at /Users/bitsofsteve/.syft-bg/token.json
Config saved to /Users/bitsofsteve/.syft-bg/config.yaml
email_approve started
email_approve: error
Error confirms missing gcp_project_id


---
## Test 3: Start / Stop / Restart Services

In [9]:
syft_bg.reset()
syft_bg.init(do_email=DO_EMAIL, token_path=str(DO_TOKEN),
             settings={"email_approve": {"gcp_project_id": GCP_PROJECT_ID}})

SERVICES = ["sync", "notify", "approve"]
syft_bg.ensure_running(services=["sync"], restart=True)
syft_bg.ensure_running(services=["notify", "approve"], restart=True)

Error stopping service notify: notify is not running
Error stopping service approve: approve is not running
Error stopping service sync: sync is not running
Reset complete: uninstalled systemd units, stopped services, cleared state, config, logs, and PID files.
Stored token at /Users/bitsofsteve/.syft-bg/token.json
Config saved to /Users/bitsofsteve/.syft-bg/config.yaml
sync started
notify started
approve started


In [10]:
status = syft_bg.status
initial_pids = {}
for svc in SERVICES:
    info = status.service_infos[svc]
    assert info.status in (ServiceStatus.RUNNING, ServiceStatus.STARTING), f"{svc}: {info.status}"
    initial_pids[svc] = info.pid
    print(f"{svc}: PID {info.pid}")
print(f"All {len(SERVICES)} services running")

sync: PID 62673
notify: PID 62674
approve: PID 62675
All 3 services running


In [11]:
# Restart approve — PID should change
syft_bg.restart("approve")
new_pid = syft_bg.status.service_infos["approve"].pid
assert new_pid != initial_pids["approve"], f"PID didn't change: {new_pid}"
print(f"approve restarted: {initial_pids['approve']} -> {new_pid}")

approve restarted: 62675 -> 64225


In [12]:
# Stop approve, others keep running
syft_bg.stop("approve")
s = syft_bg.status
assert s.service_infos["approve"].status == ServiceStatus.STOPPED
for svc in ["sync", "notify"]:
    assert s.service_infos[svc].status in (ServiceStatus.RUNNING, ServiceStatus.STARTING)
print("approve stopped, sync/notify still running")

approve stopped, sync/notify still running


In [13]:
# Start approve again
syft_bg.start("approve")
assert syft_bg.status.service_infos["approve"].status in (ServiceStatus.RUNNING, ServiceStatus.STARTING)
print("Start/Stop/Restart test passed")

Start/Stop/Restart test passed


---
## Test 4: Auto-Approve Job via API

In [14]:
def create_parameterized_job(main_file, params):
    d = Path(tempfile.mkdtemp(prefix="syftbg_test_"))
    (d / main_file).write_text(
        'import os, json\n'
        'with open("params.json") as f: params = json.load(f)\n'
        'os.makedirs("outputs", exist_ok=True)\n'
        'with open("outputs/result.json", "w") as f:\n'
        '    json.dump({"params": params, "status": "done"}, f)\n'
        'print(f"Result: {params}")\n'
    )
    (d / "params.json").write_text(json.dumps(params))
    return d

In [15]:
syft_bg.ensure_running(services=["sync"], restart=True)
syft_bg.ensure_running(services=["notify", "approve"], restart=True)
time.sleep(10)  # wait for services to be fully up
syft_bg.status

syft-bg status
========================================
  email:       stephen@openmined.org
  syftbox:     /Users/bitsofsteve/SyftBox_stephen@openmined.org
  environment: local

tokens
----------------------------------------
  notify.gmail_token_path             ready
  notify.drive_token_path             ready
  approve.drive_token_path            ready
  email_approve.gmail_token_path      ready

services
----------------------------------------
  notify           🟡 starting (PID 65792) (not installed)
  approve          🟡 starting (PID 65927) (not installed)
  email_approve    ⚪ stopped (not installed)
  sync             🟡 starting (PID 65785) (not installed)

In [28]:
syft_bg.status

syft-bg status
========================================
  email:       stephen@openmined.org
  syftbox:     /Users/bitsofsteve/SyftBox_stephen@openmined.org
  environment: local

tokens
----------------------------------------
  notify.gmail_token_path             ready
  notify.drive_token_path             ready
  approve.drive_token_path            ready
  email_approve.gmail_token_path      ready

services
----------------------------------------
  notify           🟡 starting (PID 65792) (not installed)
  approve          🟡 starting (PID 65927) (not installed)
  email_approve    ⚪ stopped (not installed)
  sync             🟡 starting (PID 65785) (not installed)

In [52]:
syft_bg.status

syft-bg status
========================================
  email:       stephen@openmined.org
  syftbox:     /Users/bitsofsteve/SyftBox_stephen@openmined.org
  environment: local

tokens
----------------------------------------
  notify.gmail_token_path             ready
  notify.drive_token_path             ready
  approve.drive_token_path            ready
  email_approve.gmail_token_path      ready

services
----------------------------------------
  notify           🟢 running (PID 65792) (not installed)
  approve          🟢 running (PID 65927) (not installed)
  email_approve    ⚪ stopped (not installed)
  sync             🟢 running (PID 65785) (not installed)

In [53]:
JOB_NAME = f"api_auto_{uuid.uuid4().hex[:8]}"
ds_client.submit_python_job(user=DO_EMAIL, code_path=str(create_parameterized_job("main.py", {"run": 1})),
                            job_name=JOB_NAME, entrypoint="main.py", force_submission=True)
print(f"Submitted: {JOB_NAME}")

📤 Submitting '/var/folders/8b/hf60m8x17hb3hb625bnlmm3m0000gn/T/syftbg_test_9ahr58xn' to stephen@openmined.org...
   Job name     : api_auto_c00ac46c

✅ Job submitted successfully!
   Status : inbox (waiting for DO to review)

⏳ Next step: wait for stephen@openmined.org to approve and run it.
   Check progress with: client.jobs
Submitted: api_auto_c00ac46c


In [ ]:
# Wait for job to appear on DO side (sync service keeps local state updated)
pending_job = None
for _ in range(24):
    pending_job = next((j for j in do_client.job_client.jobs if j.name == JOB_NAME), None)
    if pending_job:
        break
    time.sleep(5)

assert pending_job, f"DO never saw job {JOB_NAME}"
print(f"DO sees job: {JOB_NAME} ({pending_job.status})")

In [ ]:
result = syft_bg.auto_approve_job(pending_job, file_paths=["params.json"])
assert result.success, f"auto_approve_job failed: {result.error}"
print(result)

In [ ]:
# Verify rule
rules = syft_bg.list_auto_approvals()
assert JOB_NAME in rules
rule = rules[JOB_NAME]
assert {e.relative_path for e in rule.file_contents} == {"main.py"}
assert "params.json" in rule.file_paths
assert DS_EMAIL in rule.peers
assert JOB_NAME in syft_bg.status.auto_approvals
print(f"Rule '{JOB_NAME}' created and visible in status")

In [ ]:
# Wait for job to complete
for _ in range(24):
    job = next((j for j in do_client.job_client.jobs if j.name == JOB_NAME), None)
    if job and job.status == "done":
        break
    time.sleep(5)

assert job and job.status == "done", f"Job not done: {job.status if job else 'missing'}"
print(f"Job {JOB_NAME}: {job.status}")

In [ ]:
# DS fetches result
ds_job = next((j for j in ds_client.job_client.jobs if j.name == JOB_NAME), None)
assert ds_job and ds_job.status == "done"
result_data = json.loads(ds_job.output_paths[0].read_text())
assert result_data["params"]["run"] == 1
print(f"DS result: {result_data}")

In [ ]:
# Follow-up job — should auto-approve
FOLLOWUP_NAME = f"api_auto_{uuid.uuid4().hex[:8]}"
ds_client.submit_python_job(user=DO_EMAIL, code_path=str(create_parameterized_job("main.py", {"run": 2})),
                            job_name=FOLLOWUP_NAME, entrypoint="main.py", force_submission=True)
print(f"Submitted follow-up: {FOLLOWUP_NAME}")

In [ ]:
for _ in range(24):
    job2 = next((j for j in do_client.job_client.jobs if j.name == FOLLOWUP_NAME), None)
    if job2 and job2.status == "done":
        break
    time.sleep(5)

assert job2 and job2.status == "done", f"Follow-up not done: {job2.status if job2 else 'missing'}"
print(f"Follow-up {FOLLOWUP_NAME}: {job2.status} (auto-approved!)")

In [ ]:
ds_job2 = next((j for j in ds_client.job_client.jobs if j.name == FOLLOWUP_NAME), None)
assert ds_job2 and ds_job2.status == "done"
result_data2 = json.loads(ds_job2.output_paths[0].read_text())
assert result_data2["params"]["run"] == 2
print(f"DS follow-up result: {result_data2}")

---
## Test 5: Job Repr for Auto-Approved Jobs (DS Side)

In [ ]:
for label, j in [("job1", ds_job), ("job2", ds_job2)]:
    assert "[approved: auto]" in str(j), f"{label} str missing auto"
    assert "approval_method='auto'" in repr(j), f"{label} repr missing auto"
    print(f"{label}: {str(j)}")

assert "auto" in ds_job._repr_html_()
jobs_str = str(ds_client.jobs)
for name in [JOB_NAME, FOLLOWUP_NAME]:
    lines = [l for l in jobs_str.splitlines() if name in l]
    assert lines and "auto" in lines[0], f"Missing 'auto' for {name}"
assert "auto" in ds_client.jobs._repr_html_()
print("All repr tests passed")

In [ ]:
# Remove rule
r = syft_bg.remove_auto_approve(JOB_NAME)
assert r.success
assert JOB_NAME not in syft_bg.list_auto_approvals()
print(f"Rule '{JOB_NAME}' removed")

---
## Test 6: Invalid DO Token at Startup

In [ ]:
syft_bg.reset()
bad_token = Path(tempfile.mktemp(suffix=".json"))
bad_token.write_text(json.dumps({"invalid": True}))

syft_bg.init(do_email=DO_EMAIL, token_path=str(bad_token),
             settings={"email_approve": {"gcp_project_id": GCP_PROJECT_ID}})

syft_bg.ensure_running(services=["sync"])
time.sleep(3)  # wait for subprocess to fail

sync_info = syft_bg.status.service_infos["sync"]
assert sync_info.status == ServiceStatus.ERROR, f"Expected ERROR, got {sync_info.status}"
assert sync_info.error
print(f"sync: {sync_info.status.value}")
print(f"Error: {sync_info.error[:200]}...")

bad_token.unlink(missing_ok=True)
syft_bg.stop()
print("\nTest 6 passed")

---
## Debugging

In [ ]:
for svc in ("sync", "notify", "approve"):
    print(f"\n{'='*20} {svc} {'='*20}")
    syft_bg.logs(svc, n=20)

In [ ]:
syft_bg.status

---
## Cleanup

In [ ]:
syft_bg.stop()